# WeMM-Embedding-9B: all 873 official videos, 64 frames per 10-second window

Run all cells in order on a Colab **G4 (96 GB class) GPU**. This notebook downloads the pinned WeMM model and each official MP4 member automatically. It uses native WeMM video processing, BF16, SDPA, 64 uniformly sampled frames, 512-pixel longest side, and non-overlapping 10-second windows with a 2-second minimum tail. Batch 4 is tried first; a CUDA memory failure selects batch 2 or 1. If even batch 1 fails, the notebook stops with no lower-frame substitute.

Each video's vectors and metadata are saved to `MyDrive/Vecna/WeMM9B_full_10s64f_v1/` with SHA-256 verification. The notebook skips only verified completed videos. It stops beginning new videos after 7.5 hours in one Colab session, so rerun all cells in a new GPU session until the full completion report exists. The media is downloaded as needed and removed from Colab scratch after its vectors reach Drive; Drive stores the vectors, not 83 GB of MP4s. Runtime disconnection may interrupt the current video, but completed video outputs remain resumable. No retrieval index is changed.


In [ ]:
import os, sys, subprocess, pathlib, json, hashlib, base64, gzip, shutil, time, gc, zlib
import numpy as np
import torch
SESSION_STARTED = time.perf_counter()
assert torch.cuda.is_available(), 'Select a G4 GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print('GPU:', GPU_NAME, '| VRAM GB:', round(GPU_GB, 1))
assert GPU_GB >= 80, 'Use a GPU with at least 80 GB VRAM for the 64-frame canary.'
assert torch.cuda.is_bf16_supported()
assert shutil.disk_usage('/content').free >= 30 * 1024**3, 'Need 30 GB of local scratch for the pinned model.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.2.0', 'qwen-vl-utils==0.0.14', 'accelerate==1.12.0', 'decord==0.6.0', 'remotezip==0.12.3', 'huggingface-hub>=0.35,<2', 'pillow>=10'], check=True)
from google.colab import drive
drive.mount('/content/drive')
WORK = pathlib.Path('/content/wemm9b_full_64f_v1')
WORK.mkdir(parents=True, exist_ok=True)
OUT = pathlib.Path('/content/drive/MyDrive/Vecna/WeMM9B_full_10s64f_v1')
OUT.mkdir(parents=True, exist_ok=True)
print('Output:', OUT)


In [ ]:
PAYLOADS = {'corpus_manifest.json': '...', 'embed_temporal.py': 'H4sIAAAAAAAC/9U7a3ObSLbf/Su47IeAR8KSnKfvMHU9ibJxlWNnbSezNb66FIJGYi0Bw0O249V/3/PohgZkx7N1v6yrIomm+/R5v7rzl/86qIr8YB4nByLZGNl9uUyTwz3TNH+t4lVo5KKo1v58JYxSrLM091fGJg5Faoj1XIRhnCwK4zYul8bfbkVyOPx2OpyqFw4A2YvydG14XlSVVS48z4gRSGn4SZKWfhmnSbG3p8byRebnhVDPi0D9WvrFchXP1eM/ijRRv9NC/crrhcV9PYhIR/GqflXGa8E4ZX6JMBVCX+CRX5T3GSCvxi/FH5VIAlFjmVTr7N7wCyPJ9va+nXyYnnvTv19Nzy5Pzs8uDdd4MJ119tIcGPB9s+HvlL/9TUzft2K+5hcvN+Z278P04/HX0yvv8/mH6SlAMJGVB31+Die/mvXkv/02PfPen3/+fHKFSw7fHYrJuzdvw8mbt68m81E4mhyOwtfv3r2MDt9E74LxKwDy5jAw27t5F9NvJ4g5wngXTaI34s0Y1oVvx6NXr33/VTDyX0ajl29ev4xE8Ho+fzMPfXNvby8UETA8S708TUvLNoa/EAeP9gz4ywUIO6EBC0QP7Pc82wFNSlcbYdkOSFkkZXE9mUlIJHZvJTZiVVgbf1WJI6Moc+OfxlmaCIJeiPIahmZyjDeKIwPUyOAVNKJtj9N0dB5iUAYHYMQZIByluYEDRpzweqfIVnFpmQPTRrj65K1EM4yLIN2I3CMLAEzxizhwRMQODCbhqIstcLcmZBUX5TXOnrVpqGE54g6mFJatUeTHhTA+AiPP0vJjWiXhNM/T3IrMb2SLuMwIU1EQKFp/ZDw0ILemTbAYb8DmekbPyAS0A2JCg0C+WKVzy9w3NRQkmjjbiQsSKnBRrneKKoriO2eV3oochnEmgOxaRwMN/4I0KeOkEprcVoAawYOf4Bw2witTjcu2jg2zGhxJaFi4HyxBxSoLRKp+uB7NFDa8wP4BEswix88ykYQWImPrSlSACxAh41TYUi8KsViDQntzlExhhVVOns0DRYhWqV8OwEMmYXqrDaBmhUIbWMeJV/rxqh5qlKWsspW4lvPoa9boTrOZ8bNrjJB4tVk9oDajgZ6dSF1Iq7JRjKIE5sHjyBnR8+0S5C1Hf9b2bIABt2A+UGHxrJ9qNAba/JYEcclQAv3F1TnQEhEgpsTBsAe40m5AyQ3dmk5dYLBaSclfAx+9KPfXwgPc4kAUDFATA0DWnqJM+03rAhAx2FacoAwJYEFPJK0kc5LQz3P/vhaPnNPlPNnzN3Q7bMimmreuitKYC+MXYyRtFoBoW9cyBdR+CDROwLPFoQzYa1H6oV/6Eu5tHILhA9v9OwvELElXAlGiwniJDkMOglgtoNLP/WQhLIk0CBhipnDhBfHq9Usb5o2cV7axL7c5UIxgkpj5ABaW5MA9i7fZR6psxy8QHO4DrwCYLk4YDFbgkyWIgTFqSQbQH9dWWWLa4oECZBWgDv6czVb56q737gUx9EY06Ul3FKyEn3jsecB+cggk1dzKzev/Ox7+7g+/j4bvPGc4+wkDvgcfGceeDF0SuilMnjz2n5Zp2uy2Zr04uq/toyj0y3QdBx6mQ+SpFGWZfw9yCI8gYAWsmU3IJPfKARjykzDOLRmN3au8EgMOHl56Q49MYRQOjHKdAXUqm4KVyM/MynIBeLsENAEpgFEQJa7pwAqgFjZwtS2lUuX3jdpS8ggeNwpTsHELNzNvTVTHIMWsxzWrMhq+hagMOVfUdg1IuRNWiAiTDMqAKwvMNf0iiGP3o78qAC3QF9jendit9ZFzm0Ogt8z/TczmDSADqc3KD4QFRAyMJgiIu0BkpTGlL3BoDTYtkiSQKlnFyQ3CaGBLCL1Q3l6c+UXRtuu2xJPse0vg+/vkdcAVNS7oP1Lu851iBqIKfyO+W1FD6n+EuIqlP3n1mpMlXV7BskpuvCL+Lih4AIffgvMbjyYv5RenvKXcB920LIEcBmkpHw48JDYTD828zz+O3CjUTuKDKADcCJjnh1aDUdtCZNpHr9sA8G8Oa29ao0unyiDICAbYct1LZynuwnghCigY6hKCagJvnYZiZdEnZf4DeAN1Fyhb6OViExcoPX4R+MFSdN02p0jMXMy7GdVVGvjoxMmBEvA6pNKrXZk248qv64oFVJNGTLXcPDBVWslIPx6FI3N6l4kAKDF8iREt4cQbIsGnarHAgvMjaDDVVEYMtrBA+h5opsreqTpd8uwIJnvLqi5fP0XHWQy0J35WLNPSg8wrQYe4JyVAhDS8BKbQCijGaAvIh6KUWYSs50luXwY26p9etvQgdzlxAVEZgrvixfu0WoX6SsmMGjGMjZLs/3noI6B4AenIQqDhNOoASQatq70BCAkENxyCTRz0EdXAPNch9rhrUQkch26fc53dBix6D/0iKKjFG9stA+GxwQ5U2VhwR4/bLlBlsdx0r/LHrUg8xEgNUGJUeBlUrJyMy8SVE6YCEhk54JclUK5szG4VpjXUP1WXfomTBDQemxhQY4UCPIgIbrDGWMdFAQoMyl1D3jqoJlg0V+Dpl0YU50XpSEkX94XDVWdSiLy0RmThVr0YZGsWeWBKXqo+T5oHy8ZqiFmFg4sOvc3Kq5tXaj51W76dTiV393QOECwnqEIfK19/AzUKZpZ9RrR03Xz/9cOxEXNJXi/6b6Ncwtin8Whk5BXwKCdPF4PMoZQ1zqBkPjk2/vrlq9Ok/7V0DNc1TL8qU/OJSCYJilYQMDxYmhjGXwCHP/wj4+PL0bg1V4NsmPUKHvImZjcM7gikPShFmGGDCMfZst0ub616Masw5g9empMqk200mm0Pmrn+HZduhdtT62ZWmZZgZFl8B9J2sbDozcUio1b/1oOtgwGJe1zVsPTnVNmMXzdzkLVejEUN1v5U37o1J3hay7qliyBwmsXptq36CIy0tckHxv9Xfdozeqm8ZB1fTk6V2pys/YXY61RpT1XOEiXCpYVEvT9zYe6XAeYxm9wBJ+fRo6rinDLFRodlY+1H/VVLGTNg0+lXQfqHYZcAaO0pnAkTCX8HqaI00YJP2wnSZIOuw7z4669mqwWBxS8tdSj1gbJbY1PHrHBWuazW8wRs2bKaiYOWEkkULgSRj23w0+Oz97+fX9ptZOseEz21VIXfD5QEpIqQz/KoXJz74EujyGrrVJxAoQviVi3rayz/ZgNmVZNt7mxV5OmtxueMmoTc+gmqHMOi7BSMdXC21hvCJT8bK8hCGQ3NOZb+jZC9IQlsoE+Emj1LtXy+59GAcFxOsT3LU+BIYZG+MIBrWH2E2/9EO83sdgaLlClWAyQnFCWkDJD4kN3Ad5BV8CnVrr2WoLoEtpsVK7b8rLO3lyBrzAPqm5mD+s2+MemVG1rAOa/K8+izWKf5/Y7CQ5sIVVl571E2ZLWJWARgAKsVpKBWL7dn0bjGuI861zGPEMOaQKsPDoxOVZ1RSycyr9N0PQOdDqsAIy17AKQe8DYeJLStiT3NqlhqeZbW60mTAKqJBCsKFCQ4TkhC3JGqH6Q6cEP+6fbOwJCdIH2o45TzBewAYdnDXpnWPuHygtzoPE1XM813hiJI81C5T2rHX0BBJbBSyao92U0Va+6h7WxI6bjKPj0IS0ZEWKfhjckOvyxMbKrhIUWrgeQk2Xfp4pCGfwsEdlWa7EPDpc4Aqetew28PU7oDjHTAWQfiiUQF1wNq1MPB8FdYDUSsSb1S3JVWrw3U02FchTEFyl+QXBEvQFuqXEDWD4qthHndeTnbofDS+YLuNiBVSKaQBqo6ssHDo6Y+Mzei1gDLNAdiNf2wMNUhwQPEoLxz0Q+N4Dd4Iq9cIgcKlxiJKhYKNaZqwabn6ZI33eTyTYa+m52bDLb+ZuHBsPJu2pGB2wJ0QIux4QxfvxgjA7IoUZ8B8OkG6vFjxx1sQY5q/APa8F4Oqt58a7Bp+fOwrafdDL5hZ+1Wips4w1MdoyroYFriU6gjr10uRRMwFBDUFGSxaMfY7dMP7Vkw3fLJX628JjdSERPmABCPQ1J/XCo7jLIIwHcRilanTUftrjThPkEb6KAFrHcw15rbVkImvB7iMMRR9YdJRRcHkls3BWgzUkXbeh+to6/SaCgKs3tuzmoWHdFhlTrigYwmvMNcT6e7TRlLSm1IT23/gKKro38Stl9qkqyTsfCuff5wOHkE18ck/oTU2+RRFtvRcc78B1ruvaswaBFRVwmSZ/08nL1xr2jSxil1raE+RqPi0YNJZmYeSXS39k7Cdxza1YTZnaPcxGpvZeOBYEfV2pKX1rOn/+aw54OnueOzJT19aPSzSSK4jqtb6s0hcBPzBjsU3OU9Bu0TyAIPwPyC6w4e6Z6MHU60IhNVU18iyC8+saBVf+EU2CS4sTQttp/UXVWUPvd4C4B5kAFDTtAkFHjBpUZof1+F1wZJ1g1iHSgIQO69Qll68/tSFDCBEQBCMBMvSk6OuyvW2FXxkl3z1Sttjab7SkWlJWhzMgSGptKMNXEMXmlBrZnRTgaOSG/Zgm0dZ1aeMF6bR5xNkLZg5zQT1+PZrrkoNJOab2o2De2YimcJyActK9PFpyV6uvw0OAFkEdhRrZL4DgAh/xz8UJO2ddsJzQ+PHeR1l8b+OKGS0qTzCHOGeYF22sGJjW5gdEpZZ3gDAmKraLgiNybNSrO0vZ0FzJNlj+rga7JR8Z5Khjlea/OU4lqc83PLqXPS0OmKYqDGioD5AK4lihctczAJCvCUcxsKo52X9RYwqz2gzaTWJqV8Qbpex6hpO259aQvaCZdCoJOGaWrcSsXU9E6C1rEnPXKoFU/04cwmrLRmc79Eszmp+Kbqr5nay5r/CKLdWWMlrdP5/vlYfTJcWCypAV3d8W7EPTf4bYcqC4FNJ/1cSr8wtr+v1naqh6Nm623rChtSyimduszonCGTMl+VQlSCufrdOc4TDbdZcpwvKnQ0X/Apt0JRBHlMxYXreWEaeJ5c5PhhiHvSbMscDsnshggXcmCKJFznAoZ+tSrpyUoLrApEsrFMvqF1cX5+BfPr6pDMHSqtnbs08hkO2dvI/ZpKr963Hnps//OvV1++XnURgOoI3Fl40PiBA+rcDzerobqFOtyMAUO9z9rnBttjQ722Md1+hFet25D2k4CGte0+DrG+T9kFXb/4MVOR1CHqx7/HUnIRF9Mv5xpDHXKQDRNrxv6QhXxpbjfFp9Nv09NLPGMzwccuxSpzzffgsfxhIUCRMcrwUQ5DwTsB2C52Fo5xOhkPTicT87Ft2Z8MlYuSuiw73QoVrnE1hH47Oftw/pt3OX1/fvaBEBuPHlNj2IQ93p/d5PLqAozm2ZuwkxyCk5REqX2om6R2wTRB2+PjxfHn6aX3ZXrhMU24zdsf7TIkd/tc8N4lkIFwX40nT0Cm/HuIadlzIP96fPX+E0D+nSA/BRdK/yGW/n+W/Z9Pzryr45NTXQBP7cPNk6FsnjyHhg8AGAR89elievwD+LrZNtGqsdFgmVKufs2ndgN5PDbYddg265u2htTx1dX07IodCwP7keHGkDkMpSt/BtWnJ5BTeBQPiOandBpjbQPaDygyQVqR5pCDQmzd7S4+HV9+am9A55njR50ANRB3b9CK05mjB18Zj9d+nHAkBkI59OJ7umCsTaZrInyT2G1fQ6ekRd4Zbl+e7l4Dp5lN3aRugdutRJrkIVdgb+2oc+NYFWPF9VFv/qx3UbzoHj1f3mMjd3oXl+3uQGSepWqHCNNhA/6J3HjoIL01LMbafZBXnCUR2lVr6ga+gHLzxdZ2TK1dwAlxcxGjld5o8WcZcSQtzOZ8WFUDrWSYIOg3g7SMWsuutdsVze52A7t/2qoaKuKRs2hGhnaoLxg8u4EiB5qjXe0MWJYbsP8z6g8NYbvWXEcr7Z57YUUvvZqD6w4wFBCiw7kud/8bX/SgFfmqHqd8j6vVjho1V6bsbavdwM3bppVhXgPUmfFAnWu+TL9VevpPqZ3qFlCbOddH48lsC5NMDdrDbhFtZYfq4GFHRXS02OJeFODch06zaWtq0gvTRKgeXHMwim3oTITNM7Z86Fmrqeu+XyxPebDrJxLwcZgfSdJlA88dty6i7e7PdBje9NHmArYRO3bnapU6gbd+4TVotw/P2k1FPpfqebau7jx9grajo9hm5U8uY9Y00iRy8AL1RcNXazSv/Kzo8tkYSgZ0jgs6zvD6Id4etJQOdPAF/R838QJdnc4h9neIsWG82Gr6JnUOxIJ6qKhxH4gWHHqQOB4542hbaNZUdyvZZrs+lK6v9AiTivWIHQUp3rIoxUxDpMXlLf8fPhHKaAIengnctkxIouw+MBIHxuHr0ejImUTbpdm/zTKCMAvs8vj2jkeR3PMw6HqevJ3UC0wcku29fwGutFWTczgAAA==', 'wemm_adapter.py': 'H4sIAAAAAAAC/7VWTY/bNhC961cQPNGGomSD7QIxqkMKpLcNcijQg2EQXGlsE0uRCknZ2RT57x1+SJYcL5pLfbElDmfevHkzY0rpZ+HlCcjf8Pj45lP3BG0r9eHNhz/ISbZgSG9NA84ZS4RuCYwG5GxF34OtKKVFsbemI5zvBz9Y4JzIrjfW4w1tPLo32mWbXvijkk+jwRd8LIqihf0lDpe6H7xj04uSSA+dK8neig4cx6j8LHVrzvXDfX7LHYKtf7t7v9oUBD8x2NczaH5SfPBSuTFkdstP0iEuDLY3RbzSGH0C6xJcUpPtLnnCzEN8InXCkQKkIAEPmob3Wxr5orvpWDgHGFCBZslyRer65yQm+wWAKrCrW7b9h1qjgG4IHdAdLQlFOw/a4ys89C99PEzBy/HHJscpJ++3PtRjeRTv5TdQbrozw0bWM3oXDz92P3ar6BwJ7XofGZtKFtCrF94chefITa+EB9aUxJtn0PI71H8K5aAkom35ATTYmDRPrtLh6hp5KEQTqrDgKdEtO3HAZFFezTO0JXk+C3sIkG5Um90mvExOOCq0OWKKCPLuoSQWUNKoosAqT27rv+wAF2IXFh140Qovok1KIcsgQSPSEewJ8tloiA0V1JGOojrCY1RZuhudIrTRLWbUiZ4p6XxJvsuerfPdRSihFIs3K3cUPWzf7W4KLzKamhxZTaGSn5TnlraGO4HVA54uU3REYnWKlLkblL+wbCzz8M3XWRGZUVeP1Ukh6pzUa8q8InL8EYobJ0+in3g76CaWLovpNX+YBAKdiS4XDHsIEbua9h77Zr1OSS9rNul5FK6tMorUEUgrtfJw9LS46vjETR4K/GBly/3xTHe3qjwrm9SeBWOs2e1pQd6+Je9j5YJZKNyrkYqLOrMRjtpGYTTysRW9B5tGWZi+HFtDes6ZA7VHvZkWVElmI9gb2xxXl9kX7KpoFjQZvpdHl7Uxk8fSJLrE4/hdTFCydUaSaNpcD9U78vucxvB0P9mkBTIPPO6UJbTROyJhtBlaQS9D5yz9cQazwskBFnQDPCTLZoiiZqFBsxDywks17Uq2Xqf4JcEZzhvRHOF6yOW8sqPUuEEA7JJkSe7ffXj46cocpNuHMgLLblZV0NTqalSNQfLiVUa0PGEFy8KC3sS9nFPMazMVaVqt3grtUIUdzs/R5OPgzWMSTvj5Zar6uCYmQSyOq+AQRz+gT6mhZc4nGKvY5M5j+3bGI2/o+2r2/uJHmQbX3F4qbCWj1ctsOv93iy86fFT8lOv/h/4m6JK0YePXqd5PeyyexzX1iy6F97gDw0Dv8A9EGp7UtT0Kvwr6Z6sKTmJUTFZLnhXstZmQv/GvR/RU/AvUA8JpUwoAAA=='}
PAYLOAD_SHA256 = {'corpus_manifest.json': 'a7a4e66e156e2679d5555e6d9b124e15d54a08aa3d0abc738dd0ddaf407c8c48', 'embed_temporal.py': '425bbf4a26c1d3fba36990b72539b301426c39242589b30b09045a585ec60778', 'wemm_adapter.py': 'da1930fd1f00032c1fb643382172c2fb80b95d5bd3391fc60fbb981510e2059c'}
for name, packed in PAYLOADS.items():
    raw = gzip.decompress(base64.b64decode(packed))
    assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256[name], name
    destination = WORK / name
    if destination.exists():
        assert hashlib.sha256(destination.read_bytes()).hexdigest() == PAYLOAD_SHA256[name], name
    else:
        destination.write_bytes(raw)
sys.path.insert(0, str(WORK))
import embed_temporal as runner
import wemm_adapter
MANIFEST = json.loads((WORK / 'corpus_manifest.json').read_text(encoding='utf-8'))
MANIFEST_SHA = PAYLOAD_SHA256['corpus_manifest.json']
VIDEOS = MANIFEST['videos']
assert MANIFEST['schema'] == 'wemm9b-full-corpus-64f-v1'
assert len(VIDEOS) == 873 and len({v['id'] for v in VIDEOS}) == 873
assert MANIFEST['frames_per_window'] == 64 and MANIFEST['frame_side'] == 512
print('Videos:', len(VIDEOS), '| source GB:', round(sum(v['size_bytes'] for v in VIDEOS)/1e9, 2))


In [ ]:
from remotezip import RemoteZip
from decord import VideoReader, cpu
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor
from collections import deque
VIDEO_ROOT = WORK / 'videos'
VIDEO_ROOT.mkdir(exist_ok=True)
ZIP_HEADERS = {'User-Agent': 'Mozilla/5.0', 'Referer': 'https://aic-data.ledo.io.vn/'}

def sha_file(path):
    return runner.sha256_file(path)

def ensure_video(item, attempts=4):
    target = VIDEO_ROOT / item['relative_path']
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert target.stat().st_size == item['size_bytes'], item['id']
        return target
    for attempt in range(1, attempts + 1):
        partial = target.with_suffix('.partial')
        partial.unlink(missing_ok=True)
        try:
            with RemoteZip(item['archive_url'], headers=ZIP_HEADERS, timeout=180, support_suffix_range=True) as archive:
                member = archive.getinfo(item['zip_member'])
                assert member.file_size == item['size_bytes'], item['id']
                crc = 0
                with archive.open(member) as source, partial.open('wb') as dest:
                    while block := source.read(8 * 1024 * 1024):
                        dest.write(block)
                        crc = zlib.crc32(block, crc)
                assert crc == member.CRC, f'ZIP CRC mismatch: {item["id"]}'
            assert partial.stat().st_size == item['size_bytes'], item['id']
            partial.replace(target)
            return target
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt == attempts:
                raise
            time.sleep(min(2 * attempt, 8))

def prefetch(items, ahead=3):
    with ThreadPoolExecutor(max_workers=ahead) as pool:
        waiting, source = deque(), iter(items)
        def submit():
            try: item = next(source)
            except StopIteration: return
            waiting.append((item, pool.submit(ensure_video, item)))
        for _ in range(ahead): submit()
        while waiting:
            item, future = waiting.popleft()
            submit()
            yield item, future.result()


In [ ]:
from huggingface_hub import snapshot_download
MODEL_PATH = pathlib.Path(snapshot_download(repo_id=MANIFEST['model'], revision=MANIFEST['model_revision'], local_dir=str(WORK / 'model'), max_workers=4, token=False))
model, torch, attention = wemm_adapter.load_embedder(MODEL_PATH)
assert attention == 'sdpa'
print('Loaded', MANIFEST['model'], MANIFEST['model_revision'][:12])


In [ ]:
# Verify the exact 64-frame path and choose the largest fitting batch, starting at 4.
CANARY_ITEM = VIDEOS[0]
CANARY_VIDEO = ensure_video(CANARY_ITEM)
vr = VideoReader(str(CANARY_VIDEO), ctx=cpu(0), num_threads=2)
fps, nframes = float(vr.get_avg_fps()), len(vr)
assert fps > 0 and nframes > 0
bounds = runner.segment_bounds(nframes/fps, 10, 10, 2)[:4]
assert len(bounds) == 4
samples = [{'video': runner.load_segment_frames(vr, start, end, fps, nframes, 64, 512)[0]} for start, end in bounds]
del vr
probe = []
BATCH_SIZE = None
for batch in (4, 2, 1):
    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    try:
        t = time.perf_counter()
        output = model.process(samples[:batch]).detach().float().cpu().numpy()
        torch.cuda.synchronize()
        assert output.shape == (batch, 4096) and np.isfinite(output).all()
        peak = torch.cuda.max_memory_reserved() / 1e9
        record = {'batch':batch, 'seconds':time.perf_counter()-t, 'peak_reserved_gb':peak, 'gpu_gb':GPU_GB, 'status':'passed'}
        probe.append(record); print(record, flush=True)
        del output
        if peak < GPU_GB * 0.90:
            BATCH_SIZE = batch
            break
    except torch.cuda.OutOfMemoryError:
        probe.append({'batch':batch, 'status':'oom'})
        print('CUDA OOM at batch', batch, flush=True)
        torch.cuda.empty_cache(); gc.collect()
assert BATCH_SIZE is not None, 'Even batch 1 lacks safe VRAM margin at 64 frames / 512 side.'
(OUT / 'memory_canary.json').write_text(json.dumps({'manifest_sha256':MANIFEST_SHA, 'model_revision':MANIFEST['model_revision'], 'results':probe, 'selected_batch':BATCH_SIZE}, indent=2))
del samples


In [ ]:
# Start or resume the complete 873-video corpus. Safe to rerun after a Colab disconnect.
MAX_SESSION_HOURS = 7.5
RUN_ROOT = WORK / 'local_results'
RUN_ROOT.mkdir(exist_ok=True)
args = SimpleNamespace(model=MANIFEST['model'], model_revision=MANIFEST['model_revision'],
    window_seconds=10.0, stride_seconds=10.0, min_tail_seconds=2.0,
    frames_per_window=64, frame_side=512, batch_size=BATCH_SIZE,
    decode_threads=2, hash_videos=True, force=True)
signature_data = {key: MANIFEST[key] for key in ('model', 'model_revision', 'window_seconds', 'stride_seconds', 'min_tail_seconds', 'frames_per_window', 'frame_side')}
signature_data.update({'dtype':'bfloat16', 'attention':'sdpa', 'processor':'WeMM native', 'adapter_sha256':PAYLOAD_SHA256['wemm_adapter.py'], 'runner_sha256':PAYLOAD_SHA256['embed_temporal.py']})
run_signature = hashlib.sha256(json.dumps(signature_data, sort_keys=True).encode()).hexdigest()
run_meta = {**signature_data, 'run_signature':run_signature, 'manifest_sha256':MANIFEST_SHA}
config_path = OUT / 'run_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == run_meta, 'Existing output contract differs.'
else:
    config_path.write_text(json.dumps(run_meta, indent=2))

def output_paths(item):
    stem = pathlib.Path(item['relative_path']).with_suffix('')
    return OUT / 'vectors' / stem.with_suffix('.npz'), OUT / 'vectors' / stem.with_suffix('.json')

def completed(item):
    vector_path, meta_path = output_paths(item)
    if not (vector_path.exists() and meta_path.exists()): return False
    meta = json.loads(meta_path.read_text())
    assert meta['run_signature'] == run_signature and meta['manifest_sha256'] == MANIFEST_SHA, item['id']
    assert meta['video_size_bytes'] == item['size_bytes'], item['id']
    assert sha_file(vector_path) == meta['vector_sha256'], item['id']
    return True

def atomic_copy(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + '.partial')
    shutil.copyfile(source, partial)
    assert sha_file(source) == sha_file(partial), target
    partial.replace(target)

remaining = [item for item in VIDEOS if not completed(item)]
print('Verified completed:', len(VIDEOS)-len(remaining), '| remaining:', len(remaining), '| batch:', BATCH_SIZE, flush=True)
stream = prefetch(remaining, ahead=3)
processed = 0
try:
    for item, video in stream:
        if (time.perf_counter() - SESSION_STARTED) / 3600 >= MAX_SESSION_HOURS:
            print('Session cutoff reached. Rerun the notebook in a new G4 session.', flush=True)
            break
        started = time.perf_counter()
        source_sha = sha_file(video)
        count, reused = runner.process_video(video, VIDEO_ROOT, RUN_ROOT, model, torch, args, run_meta)
        assert count > 0 and not reused
        stem = pathlib.Path(item['relative_path']).with_suffix('')
        local_vector = RUN_ROOT / 'vectors' / stem.with_suffix('.npz')
        local_meta = RUN_ROOT / 'vectors' / stem.with_suffix('.json')
        with np.load(local_vector) as arrays:
            assert arrays['embeddings'].shape == (count, 4096)
            assert arrays['frame_indices'].shape == (count, 64)
            assert np.isfinite(arrays['embeddings']).all()
        metadata = json.loads(local_meta.read_text())
        assert metadata['video_sha256'] == source_sha
        metadata.update({'manifest_sha256':MANIFEST_SHA, 'vector_sha256':sha_file(local_vector),
                         'requested_batch_size':BATCH_SIZE,
                         'source_archive':item['archive_url'], 'source_zip_member':item['zip_member']})
        local_meta.write_text(json.dumps(metadata, indent=2))
        vector_path, meta_path = output_paths(item)
        atomic_copy(local_vector, vector_path)
        atomic_copy(local_meta, meta_path)
        assert completed(item)
        video.unlink(); local_vector.unlink(); local_meta.unlink()
        processed += 1
        print('SAVED', processed, '/', len(remaining), item['id'], count, 'windows', round(time.perf_counter()-started, 1), 's', flush=True)
finally:
    stream.close()
print('This session saved', processed, 'new videos.', flush=True)


In [ ]:
# Verify coverage after this session; only complete when every output is present and hash-valid.
missing = [item['id'] for item in VIDEOS if not completed(item)]
summary = {'schema':MANIFEST['schema'], 'manifest_sha256':MANIFEST_SHA, 'run_signature':run_signature,
           'verified_videos':len(VIDEOS)-len(missing), 'expected_videos':len(VIDEOS),
           'missing_video_ids':missing, 'complete':not missing}
(OUT / 'latest_coverage.json').write_text(json.dumps(summary, indent=2))
print('Verified:', summary['verified_videos'], '/', summary['expected_videos'], flush=True)
if not missing:
    summary['window_count'] = sum(json.loads(output_paths(item)[1].read_text())['segment_count'] for item in VIDEOS)
    (OUT / 'completion_full_corpus.json').write_text(json.dumps(summary, indent=2))
    print('FULL CORPUS VERIFIED:', summary['window_count'], 'windows', flush=True)
else:
    print('Resume in a new G4 session. First missing IDs:', missing[:20], flush=True)
